# GeecsDevice basics

The entry-level way to talk to a GEECS device from Python: **get**, **set**,
and **subscribe**, with errors that raise instead of returning `None`.

Requirements: the lab network (or VPN) and the standard
`~/.config/geecs_python_api/config.ini` — see the
[Getting Started tutorial](../../../tutorials/getting_started/). Run
`scripts/lab_status.sh` first if you are unsure whether the lab is reachable.

The examples use `U_S1H` (an HTU steering magnet supply) and its `Current`
variable — substitute your own device and variable names.

In [ ]:
from geecs_core import GeecsDevice

# One database query resolves the device's endpoint; no sockets are
# opened until the first get/set/subscribe. Unknown names raise
# GeecsDeviceNotFoundError immediately.
dev = GeecsDevice("U_S1H")
dev

## Read a variable

`get` blocks until the device answers (default budget 10 s) and returns the
value typed — `float`/`int` for numerics, `str` for text. A failure raises
(`GeecsCommandFailedError` for a device-reported error,
`GeecsConnectionError` for a timeout) — you never have to `None`-check.

In [ ]:
current = dev.get("Current")
print(f"Current = {current!r}  (type: {type(current).__name__})")

## Write a variable

`set` rides GEECS's native blocking convergence — it returns when the device
reports the value settled (default budget 30 s), and what it returns is the
**device's reported readback**, not an echo of what you sent.

<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>This commands real hardware. The cell below writes back the value just
read (a no-op move) — adapt with care.</p>
</div>

In [ ]:
readback = dev.set("Current", current)
print(f"device reports Current = {readback!r}")

## Subscribe to live updates

`subscribe` opens the device's TCP push stream (a few Hz). Every frame
updates `dev.state` and adds two reserved keys: `"shot number"` (the
device's frame counter) and `"connected"`. The stream auto-reconnects
through device restarts; pass `reconnect=False` to opt out.

An optional `on_update` callback receives each parsed frame. It runs on the
client's background thread — keep it quick, and **never call `get`/`set`
from inside it** (that raises a `RuntimeError` by design: blocking the
stream's own thread would deadlock). Hand work to a queue instead.

In [ ]:
import time

frames = []
dev.subscribe(["Current"], on_update=frames.append)

time.sleep(2)  # let a few frames arrive
print(f"{len(frames)} frames; latest state: {dev.state}")

`dev.state` always holds the last-known values — subscription frames and
get/set responses both feed it — so a monitoring loop can just read the
dict. Subscribing with `variables=None` subscribes the device's full
database-declared variable set.

## Clean up

`close()` releases the subscription and the UDP sockets; it is idempotent,
and using the device after close raises. For scripts, the context-manager
form handles it:

```python
with GeecsDevice("U_S1H") as dev:
    print(dev.get("Current"))
```

In [ ]:
dev.close()

## Where to go next

- Migrating a legacy `geecs_python_api` script? The
  [overview](../../overview/) has the side-by-side table.
- Need this inside an asyncio application? Skip the client and use
  `geecs_core.transport` directly — same wire protocol, no background
  thread.
- Running scans? That is the GEECS Console / Bluesky engine's job, not
  `GeecsDevice`'s.